# Preprocessing berita CNN Indonesia (adaptasi dari versi CNBC)

Alur, urutan tahap, threshold, dan parameter model **tidak diubah**. Perubahan hanya pada komponen yang bergantung bahasa/skema:

| Tahap | Versi CNBC (Inggris) | Versi ini (Indonesia) |
|---|---|---|
| Skema kolom | `full_text`, `preview_summary`, `published_raw`, `section`, `keyword_matched` | dibuat dari `content`, `description`, `date`, URL; `keyword_matched` kosong |
| Simbol mata uang | percent, dollar | persen, dolar |
| Filter bahasa | `en` | `id` |
| Stopword | NLTK english + custom | NLTK indonesian + custom (padanan) |
| Lemmatization | WordNet + POS tag | Stemmer Sastrawi (nama fungsi/kolom sama) |
| Sentimen | VADER + Loughran-McDonald | InSet (VADER dan LM hanya untuk Bahasa Inggris, LM tidak punya padanan) |
| Kata kunci GPR | daftar Inggris | terjemahan satu-satu, jumlah dan urutan sama |
| Ekstraksi TF-IDF + SVD, agregasi harian, merge, cutoff 15:00 | | sama |

**Prasyarat tambahan:** file `positive.tsv` dan `negative.tsv` dari lexicon InSet di `../data/external/inset/`.


In [17]:
import re
import html
import warnings
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


In [18]:
news_raw = pd.read_csv('../data/raw/cnn_internasional.csv')
bi_raw = pd.read_csv('../data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (9990, 7)
Kurs USD BI       : (1202, 5)


,title,date,author,description,image,url,content
0,"Viral Momen Bercanda PM Takaichi & Anwar, Netizen Nilai Tak Pantas",2026-11-24T18:00:08+07:00,NaN,Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...,https://akcdn.detik.net.id/visual/2026/11/24/pm-malaysia-anwar-ibrahim-pm-jepang-sanae-takaichi-1795508133582_169.jp...,https://www.cnnindonesia.com/internasional/20261124145552-113-1430863/viral-momen-bercanda-pm-takaichi-anwar-netizen...,Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...
1,"Malaysia Mau Kirim 10.000 Pengungsi Rohingya ke Turki, Ankara Menolak",2026-11-24T15:15:05+07:00,NaN,Pemerintah Turki membantah pengumuman Malaysia yang menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke ne...,https://akcdn.detik.net.id/visual/2026/07/28/pm-malaysia-perintahkan-tindak-tegas-pengungsi-rohingya-tanpa-izin-1785...,https://www.cnnindonesia.com/internasional/20261124143121-106-1430841/malaysia-mau-kirim-10000-pengungsi-rohingya-ke...,Malaysia menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke Turki .\r\nPemerintah Turki langsung menolak ...
2,Malaysia Bakal Pulangkan 6.800 Migran Rohingya ke Myanmar,2026-11-24T13:05:05+07:00,NaN,PM Anwar mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar bulan depan.,https://akcdn.detik.net.id/visual/2025/11/05/anwar-ibrahim-1762331055170_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20261124114805-106-1430750/malaysia-bakal-pulangkan-6800-migran-rohingya-...,Perdana Menteri Anwar Ibrahim mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar dalam ...
3,"Setara 1 Resimen, Ribuan Tentara Rusia Bunuh Diri Imbas Stres Perang",2026-11-24T17:25:26+07:00,NaN,Sebanyak 1.087 tentara Rusia dilaporkan meninggal dunia akibat bunuh diri karena stres perang sejak Moskow melancark...,https://akcdn.detik.net.id/visual/2022/11/04/tentara-rusia-bebas-dari-tahanan-ukraina-1_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20261124153911-134-1430906/setara-1-resimen-ribuan-tentara-rusia-bunuh-di...,Sebanyak 1.087 tentara Rusia dilaporkan tercatat meninggal dunia akibat bunuh diri karena stres perang sejak Moskow ...
4,"Estonia Tutup Perbatasan, Rusia Ancam Negara NATO Itu Perang Terbuka",2026-11-24T16:51:04+07:00,NaN,Estonia memutuskan menutup perbatasannya dengan Rusia di tengah perang Moskow melawan Ukraina yang belum kunjung usai.,https://akcdn.detik.net.id/visual/2026/11/24/perbatasan-estonia-rusia-1795513821122_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20261124163928-134-1430938/estonia-tutup-perbatasan-rusia-ancam-negara-na...,Estonia memutuskan menutup perbatasannya dengan Rusia di tengah perang Moskow melawan Ukraina yang belum kunjung usa...


In [19]:
news_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9990 entries, 0 to 9989
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        9990 non-null   object
 1   date         8819 non-null   object
 2   author       2058 non-null   object
 3   description  9990 non-null   object
 4   image        9990 non-null   object
 5   url          9990 non-null   object
 6   content      9990 non-null   object
dtypes: object(7)
memory usage: 546.5+ KB


In [20]:
# Parse timestamp CNN Indonesia (sudah +07:00 / WIB)
news_raw["date_dt"] = pd.to_datetime(news_raw["date"], errors="coerce")
news_raw["date_dt_wib"] = news_raw["date_dt"].dt.tz_localize(None)

news_raw["tanggal"] = news_raw["date_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_dt_wib"].dt.strftime("%I:%M %p")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_dt_wib"]].head(5)

Kolom Berita Geopolitik:


,title,tanggal,jam,date_dt_wib
0,"Viral Momen Bercanda PM Takaichi & Anwar, Netizen Nilai Tak Pantas",11/24/2026,06:00 PM,2026-11-24 18:00:08
1,"Malaysia Mau Kirim 10.000 Pengungsi Rohingya ke Turki, Ankara Menolak",11/24/2026,03:15 PM,2026-11-24 15:15:05
2,Malaysia Bakal Pulangkan 6.800 Migran Rohingya ke Myanmar,11/24/2026,01:05 PM,2026-11-24 13:05:05
3,"Setara 1 Resimen, Ribuan Tentara Rusia Bunuh Diri Imbas Stres Perang",11/24/2026,05:25 PM,2026-11-24 17:25:26
4,"Estonia Tutup Perbatasan, Rusia Ancam Negara NATO Itu Perang Terbuka",11/24/2026,04:51 PM,2026-11-24 16:51:04


In [21]:
# Penyesuaian skema dataset CNN Indonesia ke nama kolom yang dipakai pipeline.
# Tahap-tahap di bawah tidak diubah; hanya nama kolomnya yang disamakan.
news_raw["preview_summary"] = news_raw["description"]
news_raw["full_text"] = news_raw["content"]
news_raw["published_raw"] = news_raw["date"]
news_raw["section"] = news_raw["url"].str.extract(r"cnnindonesia\.com/([^/]+)/", expand=False)
news_raw["keyword_matched"] = np.nan  # tidak ada di dataset CNN; relevansi ditentukan oleh RELEVANCE_KEYWORDS saja

print("Baris tanpa tanggal (akan terbuang oleh filter rentang):", news_raw["date_dt"].isna().sum())
print("Rentang tanggal berita:", news_raw["date_dt_wib"].min(), "-", news_raw["date_dt_wib"].max())
news_raw[["title", "preview_summary", "full_text", "published_raw", "section"]].head(3)


Baris tanpa tanggal (akan terbuang oleh filter rentang): 1171
Rentang tanggal berita: 2026-01-22 14:15:35 - 2026-11-24 18:00:08


,title,preview_summary,full_text,published_raw,section
0,"Viral Momen Bercanda PM Takaichi & Anwar, Netizen Nilai Tak Pantas",Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...,Momen bercanda antara Perdana Menteri Jepang Sanae Takaichi dan Perdana Menteri Malaysia Anwar Ibrahim viral di medi...,2026-11-24T18:00:08+07:00,internasional
1,"Malaysia Mau Kirim 10.000 Pengungsi Rohingya ke Turki, Ankara Menolak",Pemerintah Turki membantah pengumuman Malaysia yang menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke ne...,Malaysia menyebut akan memukimkan kembali 10.000 pengungsi Rohingya ke Turki .\r\nPemerintah Turki langsung menolak ...,2026-11-24T15:15:05+07:00,internasional
2,Malaysia Bakal Pulangkan 6.800 Migran Rohingya ke Myanmar,PM Anwar mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar bulan depan.,Perdana Menteri Anwar Ibrahim mengatakan Malaysia akan memulangkan lebih dari 6.500 migran kembali ke Myanmar dalam ...,2026-11-24T13:05:05+07:00,internasional


In [22]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


# Data Cleaning

## 1. Filter Dataset


In [23]:
#delete duplicate rows
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_dt_wib"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 9990
Setelah filter rentang 1 September 2021 - 1 September 2026 : 6346
Setelah dedup url                  : 6346
Setelah dedup title+date_raw       : 6346


In [24]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "description", "content"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 6346
Setelah drop video/gambar/audio      : 5798


,title,date,author,description,image,url,content,date_dt,date_dt_wib,tanggal,jam,preview_summary,full_text,published_raw,section,keyword_matched
2827,Bos Bank of America Erin Piacenti Tewas Ditusuk di Times Square AS,2026-09-01T23:43:46+07:00,Aljazeera,Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square.,https://akcdn.detik.net.id/visual/2020/06/22/ilustrasi-penusukan-2_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901233948-134-1398948/bos-bank-of-america-erin-piacenti-tewas-ditusu...,"Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square, New Yo...",2026-09-01 23:43:46+07:00,2026-09-01 23:43:46,09/01/2026,11:43 PM,Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square.,"Vice President dari Bank of America Erin Piacenti tewas karena jadi korban penusukan di kawasan Times Square, New Yo...",2026-09-01T23:43:46+07:00,internasional,NaN
2831,"Presiden Iran Bertemu Putin, Ucap Terima Kasih atas Sikap Rusia",2026-09-01T20:40:29+07:00,NaN,"Bertemu di sela KTT SCO di Kyrgyzstan, Presiden Iran Pezeshkian sampaikan terima kasih atas sikap Rusia ke Putin.",https://akcdn.detik.net.id/visual/2026/09/01/sco-summitkyrgyzstan-1788269721227_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901202732-120-1398902/presiden-iran-bertemu-putin-ucap-terima-kasih-...,"Presiden Iran Masoud Pezeshkian menyampaikan terima Presiden Vladimir Putin dan sikap Rusia, kala Iran jadi target s...",2026-09-01 20:40:29+07:00,2026-09-01 20:40:29,09/01/2026,08:40 PM,"Bertemu di sela KTT SCO di Kyrgyzstan, Presiden Iran Pezeshkian sampaikan terima kasih atas sikap Rusia ke Putin.","Presiden Iran Masoud Pezeshkian menyampaikan terima Presiden Vladimir Putin dan sikap Rusia, kala Iran jadi target s...",2026-09-01T20:40:29+07:00,internasional,NaN
2833,"Kasus Jeju, Polisi Korsel Dicokok Gegara Palsukan Laporan Orang Hilang",2026-09-01T20:00:02+07:00,NaN,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.,https://akcdn.detik.net.id/visual/2023/05/30/ilustrasi-penangkapan-3_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901180103-113-1398848/kasus-jeju-polisi-korsel-dicokok-gegara-palsuk...,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.\r\n...,2026-09-01 20:00:02+07:00,2026-09-01 20:00:02,09/01/2026,08:00 PM,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.,Seorang polisi Korea Selatan ditangkap setelah memalsukan penanganan puluhan laporan orang hilang di Pulau Jeju.\r\n...,2026-09-01T20:00:02+07:00,internasional,NaN
2834,"Perang Lawan AS, Iran Klaim Kembangkan Sistem Pertahanan Udara Terbaru",2026-09-01T19:45:45+07:00,NaN,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,https://akcdn.detik.net.id/visual/2019/06/26/3e6d3647-1737-466a-9b93-e8bf53cb3325_169.jpeg?w=1200,https://www.cnnindonesia.com/internasional/20260901173114-134-1398843/perang-lawan-as-iran-klaim-kembangkan-sistem-p...,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,2026-09-01 19:45:45+07:00,2026-09-01 19:45:45,09/01/2026,07:45 PM,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,Iran mengeklaim tengah mengembangkan sistem pertahanan udara terbaru di tengah perang melawan Amerika Serikat dan Is...,2026-09-01T19:45:45+07:00,internasional,NaN
2835,Kedubes Rusia Buka Suara soal Dugaan 8 WNI Jadi Tentara Bayaran,2026-09-01T19:23:24+07:00,NaN,Dubes Rusia di Indonesia pastikan pihaknya tidak melakukan rekrutmen dalam bentuk apapun untuk Angkatan Bersenjata R...,https://akcdn.detik.net.id/visual/2024/10/2

In [25]:
# Panjang text minimal 350

effective_text = news["content"].fillna(news["description"])
news = news.loc[effective_text.apply(word_count) >= 350]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 1818


In [26]:
#Filter keyword yang relevan dengan geopolitik dan ekonomi
RELEVANCE_KEYWORDS = ["geopolitik", "konflik", "perang", "sanksi", "tarif", "embargo", "diplomasi", "nato", "brics", "opec", "g7", "militer", "rudal", "gencatan senjata"]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 608
Setelah filter kategori/section     : 608


,title,date,author,description,image,url,content,date_dt,date_dt_wib,tanggal,jam,preview_summary,full_text,published_raw,section,keyword_matched
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,2026-09-01T13:30:00+07:00,NaN,Pulau Larak menjadi salah satu titik strategis yang menjadi target perang Amerika Serikat dan Iran.,https://akcdn.detik.net.id/visual/2026/08/31/pulau-larak-iran-1788149830342_169.png?w=1200,https://www.cnnindonesia.com/internasional/20260901074836-120-1398568/kenapa-pulau-larak-penting-hingga-jadi-target-...,Pulau Larak menjadi salah satu titik strategis yang menjadi target perang Amerika Serikat dan Iran . Pulau kecil mil...,2026-09-01 13:30:00+07:00,2026-09-01 13:30:00,09/01/2026,01:30 PM,Pulau Larak menjadi salah satu titik strategis yang menjadi target perang Amerika Serikat dan Iran.,Pulau Larak menjadi salah satu titik strategis yang menjadi target perang Amerika Serikat dan Iran . Pulau kecil mil...,2026-09-01T13:30:00+07:00,internasional,NaN
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat",2026-09-01T12:30:00+07:00,NaN,Sejumlah mantan komandan militer senior dan pejabat keamanan Israel mengecam meningkatnya kekerasan terhadap warga P...,https://akcdn.detik.net.id/visual/2026/07/25/militer-israel-operasi-besar-besaran-di-tepi-barat-1784981863672_169.jp...,https://www.cnnindonesia.com/internasional/20260901111735-120-1398654/eks-jenderal-israel-prihatin-akui-pembersihan-...,Sejumlah mantan komandan militer senior dan pejabat keamanan Israel mengecam meningkatnya kekerasan terhadap warga P...,2026-09-01 12:30:00+07:00,2026-09-01 12:30:00,09/01/2026,12:30 PM,Sejumlah mantan komandan militer senior dan pejabat keamanan Israel mengecam meningkatnya kekerasan terhadap warga P...,Sejumlah mantan komandan militer senior dan pejabat keamanan Israel mengecam meningkatnya kekerasan terhadap warga P...,2026-09-01T12:30:00+07:00,internasional,NaN
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,2026-09-01T11:00:40+07:00,NaN,"Israel dan Finlandia diam-diam memperpanjang kerja sama pertahanan, meski tuai tekanan dari kelompok oposisi.",https://akcdn.detik.net.id/visual/2014/12/08/20578000-573d-4c6a-ac09-20d605b198f6_169.jpg?w=1200,https://www.cnnindonesia.com/internasional/20260831203845-134-1398484/israel-finlandia-diam-diam-perpanjang-kerja-sa...,Israel dan Finlandia diam-diam memperpanjang kerja sama pertahanan bilateral mereka.\r\nKesepakatan tersebut memperk...,2026-09-01 11:00:40+07:00,2026-09-01 11:00:40,09/01/2026,11:00 AM,"Israel dan Finlandia diam-diam memperpanjang kerja sama pertahanan, meski tuai tekanan dari kelompok oposisi.",Israel dan Finlandia diam-diam memperpanjang kerja sama pertahanan bilateral mereka.\r\nKesepakatan tersebut memperk...,2026-09-01T11:00:40+07:00,internasional,NaN


In [27]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 0
Jumlah baris setelah drop NaN: 608
Sisa NaN pada full_text: 0


## 2. Cleaning Data News

In [28]:
#menggabungkan title, preview, dan full_text

# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * 2)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 608


,title,combined_raw
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran? Kenapa Pulau Larak Penting hingga Jadi Target Peran...
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat","Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat Eks Jenderal Israel Prihatin, Akui Pembersihan Et..."
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer Israel & Finlandia Diam-diam Perpanjang Kerja Sama Milite...


In [29]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

Jumlah jenis simbol unik pada combined_raw: 24


,simbol,nama_unicode,jumlah
0,",",COMMA,18288
1,.,FULL STOP,17333
2,"""",QUOTATION MARK,6235
3,-,HYPHEN-MINUS,4565
4,(,LEFT PARENTHESIS,2149
5,),RIGHT PARENTHESIS,2148
6,/,SOLIDUS,975
7,',APOSTROPHE,573
8,:,COLON,405
9,?,QUESTION MARK,329


In [30]:
#cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "persen",
    "$": "dolar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran? Kenapa Pulau Larak Penting hingga Jadi Target Peran...,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran Kenapa Pulau Larak Penting hingga Jadi Target Perang...
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat Eks Jenderal Israel Prihatin, Akui Pembersihan Et...",Eks Jenderal Israel Prihatin Akui Pembersihan Etnis di Tepi Barat Eks Jenderal Israel Prihatin Akui Pembersihan Etni...
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer Israel & Finlandia Diam-diam Perpanjang Kerja Sama Milite...,Israel Finlandia Diam diam Perpanjang Kerja Sama Militer Israel Finlandia Diam diam Perpanjang Kerja Sama Militer Is...


In [31]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran Kenapa Pulau Larak Penting hingga Jadi Target Perang...,kenapa pulau larak penting hingga jadi target perang as vs iran kenapa pulau larak penting hingga jadi target perang...
1,Eks Jenderal Israel Prihatin Akui Pembersihan Etnis di Tepi Barat Eks Jenderal Israel Prihatin Akui Pembersihan Etni...,eks jenderal israel prihatin akui pembersihan etnis di tepi barat eks jenderal israel prihatin akui pembersihan etni...
2,Israel Finlandia Diam diam Perpanjang Kerja Sama Militer Israel Finlandia Diam diam Perpanjang Kerja Sama Militer Is...,israel finlandia diam diam perpanjang kerja sama militer israel finlandia diam diam perpanjang kerja sama militer is...


In [32]:
#memastikan bahasanya indonesia semua
TARGET_LANG = "id"
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == TARGET_LANG].reset_index(drop=True)
print(f"\nSetelah filter bahasa Indonesia (id): {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa terdeteksi:
detected_lang
id    608
Name: count, dtype: int64

Setelah filter bahasa Indonesia (id): 608 (dibuang 0 baris)


,lower_text,detected_lang
0,kenapa pulau larak penting hingga jadi target perang as vs iran kenapa pulau larak penting hingga jadi target perang...,id
1,eks jenderal israel prihatin akui pembersihan etnis di tepi barat eks jenderal israel prihatin akui pembersihan etni...,id
2,israel finlandia diam diam perpanjang kerja sama militer israel finlandia diam diam perpanjang kerja sama militer is...,id


## 3. Cleaning Data Kurs

In [33]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

(1202, 8)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28


In [34]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03


# Preprocessing

In [35]:
#stopword removal
# Padanan CUSTOM_STOPWORDS versi CNBC: nama sumber, kata ajakan baca, kata kerja pelaporan (said/says), angka kata.
CUSTOM_STOPWORDS = {
    "cnn", "indonesia", "cnnindonesia", "artikel", "baca", "juga",
    "dilansir", "dikutip", "kata", "katanya", "mengatakan", "ujar", "ujarnya",
    "ucap", "ungkap", "tutur", "satu", "dua",
}
STOPWORDS_ID = set(stopwords.words("indonesian")) | CUSTOM_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi lalu buang stopword Bahasa Indonesia & token 1 karakter.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [tok for tok in tokens if tok not in STOPWORDS_ID and len(tok) > 1]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

Rata-rata jumlah kata sebelum stopword removal : 530.5
Rata-rata jumlah kata sesudah stopword removal  : 318.2


,lower_text,text_no_stopwords
0,kenapa pulau larak penting hingga jadi target perang as vs iran kenapa pulau larak penting hingga jadi target perang...,pulau larak target perang as vs iran pulau larak target perang as vs iran pulau larak salah titik strategis target p...
1,eks jenderal israel prihatin akui pembersihan etnis di tepi barat eks jenderal israel prihatin akui pembersihan etni...,eks jenderal israel prihatin akui pembersihan etnis tepi barat eks jenderal israel prihatin akui pembersihan etnis t...
2,israel finlandia diam diam perpanjang kerja sama militer israel finlandia diam diam perpanjang kerja sama militer is...,israel finlandia diam diam perpanjang kerja militer israel finlandia diam diam perpanjang kerja militer israel finla...


In [36]:
#lemmatization -> stemming Bahasa Indonesia (Sastrawi)
# WordNet + POS tagger hanya untuk Bahasa Inggris. Untuk Bahasa Indonesia padanannya adalah stemmer Sastrawi.
# Nama fungsi/kolom (lemmatize_tokens, tokens_lemmatized, text_lemmatized) sengaja dipertahankan
# supaya seluruh tahap setelahnya tidak berubah.
stemmer = StemmerFactory().create_stemmer()
_stem_cache = {}


def stem_token(token: str) -> str:
    # Cache per kata unik: Sastrawi lambat bila dipanggil berulang untuk kata yang sama.
    if token not in _stem_cache:
        _stem_cache[token] = stemmer.stem(token)
    return _stem_cache[token]


def lemmatize_tokens(tokens: list) -> list:
    if not tokens:
        return []
    return [stem_token(token) for token in tokens]


news["tokens_lemmatized"] = news["tokens_no_stopwords"].apply(lemmatize_tokens)
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

news[["text_no_stopwords", "text_lemmatized"]].head(3)


,text_no_stopwords,text_lemmatized
0,pulau larak target perang as vs iran pulau larak target perang as vs iran pulau larak salah titik strategis target p...,pulau larak target perang as vs iran pulau larak target perang as vs iran pulau larak salah titik strategis target p...
1,eks jenderal israel prihatin akui pembersihan etnis tepi barat eks jenderal israel prihatin akui pembersihan etnis t...,eks jenderal israel prihatin aku bersih etnis tepi barat eks jenderal israel prihatin aku bersih etnis tepi barat ma...
2,israel finlandia diam diam perpanjang kerja militer israel finlandia diam diam perpanjang kerja militer israel finla...,israel finlandia diam diam panjang kerja militer israel finlandia diam diam panjang kerja militer israel finlandia d...


# Penyelarasan Data

In [37]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

# versi naive WIB (tanpa tz) untuk alignment & merge, sama seperti Tanggal_dt di data kurs
news['date_raw_dt'] = news['published_raw'].dt.tz_localize(None)


In [38]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

In [39]:
def align_to_trading_date(dt):
    # Step 1: kalau lewat cutoff 15:15:00, geser ke hari berikutnya
    cutoff = dt.replace(hour=15, minute=0, second=0, microsecond=0)
    if dt > cutoff:
        dt = dt + pd.Timedelta(days=1)

    # Step 2: kalau hasilnya jatuh di weekend, geser ke Senin
    if dt.weekday() == 5:      # Sabtu
        dt = dt + pd.Timedelta(days=2)
    elif dt.weekday() == 6:    # Minggu
        dt = dt + pd.Timedelta(days=1)

    return dt.normalize()  # buang jam, sisakan tanggal saja

news['aligned_trading_date'] = news['date_raw_dt'].apply(align_to_trading_date)
news

,title,date,author,description,image,url,content,date_dt,date_dt_wib,tanggal,...,normalized_text,lower_text,detected_lang,tokens_no_stopwords,text_no_stopwords,tokens_lemmatized,text_lemmatized,time,date_raw_dt,aligned_trading_date
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,2026-09-01,NaN,Pulau Larak menjadi salah satu titik strategis yang menjadi target perang Amerika Serikat dan Iran.,https://akcdn.detik.net.id/visual/2026/08/31/pulau-larak-iran-1788149830342_169.png?w=1200,https://www.cnnindonesia.com/internasional/20260901074836-120-1398568/kenapa-pulau-larak-penting-hingga-jadi-target-...,Pulau Larak menjadi salah satu titik strategis yang menjadi target perang Amerika Serikat dan Iran . Pulau kecil mil...,2026-09-01 13:30:00+07:00,2026-09-01 13:30:00,09/01/2026,...,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran Kenapa Pulau Larak Penting hingga Jadi Target Perang...,kenapa pulau larak penting hingga jadi target perang as vs iran kenapa pulau larak penting hingga jadi target perang...,id,"[pulau, larak, target, perang, as, vs, iran, pulau, larak, target, perang, as, vs, iran, pulau, larak, salah, titik,...",pulau larak target perang as vs iran pulau larak target perang as vs iran pulau larak salah titik strategis target p...,"[pulau, larak, target, perang, as, vs, iran, pulau, larak, target, perang, as, vs, iran, pulau, larak, salah, titik,...",pulau larak target perang as vs iran pulau larak target perang as vs iran pulau larak salah titik strategis target p...,13:30:00,2026-09-01 13:30:00,2026-09-01
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat",2026-09-01,NaN,Sejumlah mantan komandan militer senior dan pejabat keamanan Israel mengecam meningkatnya kekerasan terhadap warga P...,https://akcdn.detik.net.id/visual/2026/07/25/militer-israel-operasi-besar-besaran-di-tepi-barat-1784981863672_169.jp...,https://www.cnnindonesia.com/internasional/20260901111735-120-1398654/eks-jenderal-israel-prihatin-akui-pembersihan-...,Sejumlah mantan komandan militer senior dan pejabat keamanan Israel mengecam meningkatnya kekerasan terhadap warga P...,2026-09-01 12:30:00+07:00,2026-09-01 12:30:00,09/01/2026,...,Eks Jenderal Israel Prihatin Akui Pembersihan Etnis di Tepi Barat Eks Jenderal Israel Prihatin Akui Pembersihan Etni...,eks jenderal israel prihatin akui pembersihan etnis di tepi barat eks jenderal israel prihatin akui pembersihan etni...,id,"[eks, jenderal, israel, prihatin, akui, pembersihan, etnis, tepi, barat, eks, jenderal, israel, prihatin, akui, pemb...",eks jenderal israel prihatin akui pembersihan etnis tepi barat eks jenderal israel prihatin akui pembersihan etnis t...,"[eks, jenderal, israel, prihatin, aku, bersih, etnis, tepi, barat, eks, jenderal, israel, prihatin, aku, bersih, etn...",eks jenderal israel prihatin aku bersih etnis tepi barat eks jenderal israel prihatin aku bersih etnis tepi barat ma...,12:30:00,2026-09-01 12:30:00,2026-09-01
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,2026-09-01,NaN,"Israel dan Finlandia diam-diam memperpanjang kerja sama pertahanan, meski tuai tekanan dari kelompok oposisi.",https://akcdn.detik.net.id/visual/2014/12/08/20578000-573d-4c6a-ac09-20d605b198f6_169.jpg?w=1200,https://www.cnnindonesia.com/internasional/20260831203845-134-1398484/israel-finlandia-diam-diam-perpanjang-kerja-sa...,Israel dan Finlandia diam-diam memperpanjang kerja sama pertahanan bilateral mereka.\r\nKesepakatan tersebut memperk...,2026-09-01 11:00:40+07:00,2026-09-01 11:00:40,09/01/2026,...,Israel Finlandia Diam diam Perpanjang Kerja Sama Militer Israel Finlandia Diam diam Perpanjang Kerja Sama Militer Is...,israel finlandia diam diam perpanjang kerja sama militer israel finlandia diam diam perpanjang kerja sama militer is...,id,"[israel, finlandia, diam, diam, perpanjang, kerja, militer, israel, finlandia, diam, diam, perpanjang, kerja, milite...",israel finlandia diam diam perpanjang kerja militer israel finlandia diam diam per

In [40]:
daily_news = news.groupby('aligned_trading_date').agg(
    news_count=('title', 'count'),
    all_titles=('title', lambda x: ' | '.join(x)),
    all_text_lemmatized=('text_lemmatized', lambda x: ' | '.join(x))
).reset_index()

In [41]:
merged_data = pd.merge(
    bi_raw,
    daily_news,
    left_on='Tanggal_dt',                   
    right_on='aligned_trading_date',   
    how='left'
)

merged_data

,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,aligned_trading_date,news_count,all_titles,all_text_lemmatized
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01,2026-09-01,5.0,"Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran? | Eks Jenderal Israel Prihatin, Akui Pembersihan Et...",pulau larak target perang as vs iran pulau larak target perang as vs iran pulau larak salah titik strategis target p...
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,2026-08-31,6.0,"Xi Jinping, Putin & Erdogan Akan Bertemu di Tengah Sanksi AS ke Iran | Alasan AS Bombardir Pulau Larak Iran dalam Se...",xi jinping putin erdogan temu sanksi as iran xi jinping putin erdogan temu sanksi as iran xi jinping vladimir putin ...
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,2026-08-28,2.0,"Mampukah Iran Bertahan dari Tekanan Sanksi Ekonomi Total AS? | Sewot dengan Kanada, Trump Ubah Nama Danau Ontario Ja...",iran tahan tekan sanksi ekonomi total as iran tahan tekan sanksi ekonomi total as amerika serikat lancar jurus tekan...
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,NaT,NaN,NaN,NaN
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM,2026-08-26,2026-08-26,7.0,Ahli Perkirakan AS Butuh 3 Tahun Bikin Rudal usai Gempur Iran | Dewan Perdamaian Tekan Hamas Setop Terbangkan Layang...,ahli kira as butuh bikin rudal gempur iran ahli kira as butuh bikin rudal gempur iran jabat militer as ingat operasi...
...,...,...,...,...,...,...,...,...,...,...,...,...
1197,1198,1,14310.20,14167.81,2021-09-07,09/07/2021,12:00 AM,2021-09-07,NaT,NaN,NaN,NaN
1198,1199,1,14332.31,14189.70,2021-09-06,09/06/2021,12:00 AM,2021-09-06,NaT,NaN,NaN,NaN
1199,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03,NaT,NaN,NaN,NaN
1200,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02,NaT,NaN,NaN,NaN


In [42]:
# --- CEK HASIL CUTOFF (pakai date_raw_dt yang sudah WIB) ---
news['jam'] = news['date_raw_dt'].dt.time
news['weekday'] = news['date_raw_dt'].dt.day_name()
news['lewat_cutoff'] = news['date_raw_dt'].dt.time > pd.Timestamp("15:15:00").time()

# Tampilkan berita yang persis di sekitar jam cutoff (14:50 - 15:30) untuk dicek manual
mask_sekitar_cutoff = news['jam'].apply(lambda t: (t.hour == 15 and t.minute <= 30) or (t.hour == 14 and t.minute >= 50))
cek_cutoff = news.loc[mask_sekitar_cutoff, ['title', 'date_raw_dt', 'jam', 'weekday', 'lewat_cutoff', 'aligned_trading_date']]
print(cek_cutoff.sort_values('date_raw_dt').to_string(index=False))

# Cek kasus weekend shifting (berita Jumat setelah cutoff -> harus jadi Senin)
mask_weekend_case = news['weekday'].isin(['Friday', 'Saturday', 'Sunday'])
cek_weekend = news.loc[mask_weekend_case, ['title', 'date_raw_dt', 'weekday', 'lewat_cutoff', 'aligned_trading_date']]
print(cek_weekend.sort_values('date_raw_dt').to_string(index=False))

                                                                 title         date_raw_dt      jam   weekday  lewat_cutoff aligned_trading_date
  Filipina, AS, dan Jepang Latihan Militer di Dekat Laut China Selatan 2026-02-27 14:55:51 14:55:51    Friday         False           2026-02-27
   Negosiasi Nuklir Iran-AS Berakhir Buntu Lagi, Perang di Depan Mata? 2026-02-27 15:30:38 15:30:38    Friday          True           2026-03-02
       Trump Jadi Kompor, Sebut Serangan AS Peluang Rakyat Iran Kudeta 2026-02-28 15:25:15 15:25:15  Saturday          True           2026-03-02
       Beda Nasib: Kapal Indonesia Masih Aman, Kargo Tetangga RI Dibom 2026-03-12 15:20:30 15:20:30  Thursday          True           2026-03-13
      Kota Bawah Tanah Tempat Rudal Iran Disimpan, Bagaimana Sekarang? 2026-03-13 15:01:32 15:01:32    Friday         False           2026-03-16
                Korea Utara Tembakkan 10 Rudal Balistik ke Laut Jepang 2026-03-14 15:25:33 15:25:33  Saturday          True       

In [43]:
tanggal_dicek = cek_cutoff['aligned_trading_date'].unique()

for tgl in tanggal_dicek:
    subset = merged_data[merged_data['aligned_trading_date'] == tgl]
    print(f"\n=== Trading date: {tgl.date()} ===")
    print(f"Jumlah berita ter-merge: {subset['news_count'].values}")
    # tampilkan judul2 berita yg masuk ke tanggal ini dari df news asli
    judul_masuk = news[news['aligned_trading_date'] == tgl][['title', 'published_raw']]
    print(judul_masuk.sort_values('published_raw').to_string(index=False))


=== Trading date: 2026-08-26 ===
Jumlah berita ter-merge: [7.]
                                                                 title             published_raw
 Menang Perang Tak Melulu Ditentukan Keunggulan Militer, Ini Kata Ahli 2026-08-25 15:00:35+07:00
Trump Mau Patok Harga Visa Kerja AS Rp1,8 Miliar dari Semula Rp53 Juta 2026-08-25 17:50:10+07:00
       6 Bulan Perang, 20 Orang Tewas dalam 68 Insiden di Selat Hormuz 2026-08-26 01:00:24+07:00
                         Kanada Siapkan Balasan untuk Tarif Baru Trump 2026-08-26 03:00:38+07:00
                 Trump Lempar Ucapan Kasar ke PM Ontario, Ini Sebabnya 2026-08-26 07:00:00+07:00
 Dewan Perdamaian Tekan Hamas Setop Terbangkan Layang-layang ke Israel 2026-08-26 07:35:33+07:00
         Ahli Perkirakan AS Butuh 3 Tahun Bikin Rudal usai Gempur Iran 2026-08-26 12:05:25+07:00

=== Trading date: 2026-08-13 ===
Jumlah berita ter-merge: [4.]
                                                                 title             published_raw

# Ekstraksi Fitur NLP


In [44]:
from __future__ import annotations

import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 1. Sentimen berbasis Lexicon

In [45]:
# Lexicon sentimen Bahasa Indonesia: InSet (Koto & Rahmaningtyas, 2017)
# Pengganti VADER (general-purpose) dan Loughran-McDonald (finance-specific) yang hanya untuk Bahasa Inggris.
# Letakkan positive.tsv dan negative.tsv di folder berikut (kolom: word, weight).
INSET_DIR = Path("../data/external/inset")
INSET_NEGATIONS = {"tidak", "tak", "bukan", "belum", "jangan", "tanpa"}


def _load_inset_lexicon(inset_dir: Path = INSET_DIR) -> dict:
    lexicon = {}
    for fname, sign in (("positive.tsv", 1), ("negative.tsv", -1)):
        path = Path(inset_dir) / fname
        if not path.exists():
            raise FileNotFoundError(f"File lexicon InSet tidak ditemukan: {path}")
        tmp = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1], names=["word", "weight"], dtype=str)
        tmp["weight"] = pd.to_numeric(tmp["weight"], errors="coerce")
        tmp = tmp.dropna()  # baris header ikut terbuang di sini
        for word, weight in zip(tmp["word"].str.strip().str.lower(), tmp["weight"].abs()):
            lexicon[word] = sign * float(weight)
    return lexicon


_inset = _load_inset_lexicon()


def compute_inset_sentiment(df: pd.DataFrame, text_col: str = "lower_text") -> pd.DataFrame:
    # Skor bag-of-words berbobot; kata sentimen yang didahului negasi dibalik tanda (setara negasi pada VADER).
    def _score(text: str) -> pd.Series:
        tokens = text.split() if isinstance(text, str) else []
        pos = neg = 0.0
        for i, tok in enumerate(tokens):
            w = _inset.get(tok)
            if w is None:
                continue
            if i > 0 and tokens[i - 1] in INSET_NEGATIONS:
                w = -w
            if w > 0:
                pos += w
            else:
                neg += -w
        n_tokens = max(len(tokens), 1)
        total = pos - neg
        return pd.Series({
            "inset_pos": pos / n_tokens,
            "inset_neg": neg / n_tokens,
            "inset_compound": total / np.sqrt(total ** 2 + 15),     # normalisasi ala VADER (alpha=15)
            "inset_polarity": total / (pos + neg) if (pos + neg) > 0 else 0.0,  # (P-N)/(P+N) ala LM polarity
        })

    scores = df[text_col].fillna("").apply(_score)
    return pd.concat([df.reset_index(drop=True), scores.reset_index(drop=True)], axis=1)


# 3. Indeks risiko geopolitik berbasis keyword density 
# Terjemahan satu-satu dari daftar versi Inggris (jumlah dan urutan kata kunci sama)
DEFAULT_GPR_KEYWORDS = [
    "geopolitik", "risiko geopolitik", "ketegangan geopolitik",
    "fragmentasi geopolitik", "konflik bersenjata", "konflik global",
    "konflik internasional", "ketegangan militer", "keamanan nasional",
    "sanksi", "perang dagang", "tarif", "embargo", "kontrol ekspor",
    "proteksionisme", "gangguan rantai pasok", "diplomasi",
    "kebijakan luar negeri", "nato", "brics", "opec", "g7",
]

def compute_gpr_density(
    df: pd.DataFrame,
    text_col: str = "lower_text",
    keywords: list[str] | None = None,
) -> pd.DataFrame:

    kw = keywords or DEFAULT_GPR_KEYWORDS
    pattern = re.compile("|".join(re.escape(k) for k in kw))

    def _count(text: str) -> int:
        if not isinstance(text, str) or not text:
            return 0
        return len(pattern.findall(text))

    hits = df[text_col].apply(_count)
    word_count = df[text_col].fillna("").str.split().apply(len).replace(0, 1)

    out = df.copy()
    out["gpr_hits"] = hits
    out["gpr_density"] = hits / word_count
    return out

# 4. Vectorization klasik: TF-IDF + SVD (LSA) untuk reduksi dimensi
@dataclass
class TfidfSvdExtractor:
    max_features: int = 5000
    n_components: int = 75
    ngram_range: tuple[int, int] = (1, 2)
    min_df: int = 5
    random_state: int = 42

    _vectorizer: TfidfVectorizer = field(init=False, default=None, repr=False)
    _svd: TruncatedSVD = field(init=False, default=None, repr=False)
    _is_fitted: bool = field(init=False, default=False, repr=False)

    def fit(self, texts: pd.Series) -> "TfidfSvdExtractor":
        self._vectorizer = TfidfVectorizer(
            max_features=self.max_features,
            ngram_range=self.ngram_range,
            min_df=self.min_df,
            sublinear_tf=True,
        )
        tfidf_matrix = self._vectorizer.fit_transform(texts.fillna(""))

        # n_components dibatasi supaya tidak melebihi jumlah fitur/dokumen train
        n_comp = min(self.n_components, tfidf_matrix.shape[1] - 1, tfidf_matrix.shape[0] - 1)
        self._svd = TruncatedSVD(n_components=max(n_comp, 2), random_state=self.random_state)
        self._svd.fit(tfidf_matrix)
        self._is_fitted = True
        return self

    def transform(self, texts: pd.Series) -> pd.DataFrame:
        if not self._is_fitted:
            raise RuntimeError("Panggil .fit(train_texts) dulu sebelum .transform().")
        tfidf_matrix = self._vectorizer.transform(texts.fillna(""))
        reduced = self._svd.transform(tfidf_matrix)
        cols = [f"tfidf_svd_{i}" for i in range(reduced.shape[1])]
        return pd.DataFrame(reduced, columns=cols, index=texts.index)

    @property
    def explained_variance_ratio_(self) -> np.ndarray:
        return self._svd.explained_variance_ratio_


# 5. Agregasi fitur level-artikel -> level-harian (join key: aligned_trading_date)
def aggregate_daily_features(
    article_df: pd.DataFrame,
    date_col: str = "aligned_trading_date",
) -> pd.DataFrame:
    sentiment_cols = [c for c in article_df.columns if c.startswith("inset_")]
    svd_cols = [c for c in article_df.columns if c.startswith("tfidf_svd_")]
    mean_cols = sentiment_cols + svd_cols + (["gpr_density"] if "gpr_density" in article_df else [])

    agg_dict = {c: "mean" for c in mean_cols}
    if "gpr_hits" in article_df.columns:
        agg_dict["gpr_hits"] = "sum"

    daily = article_df.groupby(date_col).agg(agg_dict)
    daily["news_count"] = article_df.groupby(date_col).size()
    return daily.reset_index()

# 1.1 Fitur Sentimen (InSet Lexicon)

In [46]:
news_feat = compute_inset_sentiment(news, text_col='text_lemmatized')

news_base = compute_inset_sentiment(news, text_col='lower_text')


In [47]:
print(f"Jumlah baris feat: {len(news_feat)}")
news_feat[['title', 'inset_compound', 'inset_polarity']].head(5)

# Hasil sentimen InSet (teks lemmatized/stem)
inset_feat_result = news_feat[
    [
        "title",
        "inset_neg",
        "inset_pos",
        "inset_compound",
        "inset_polarity",
    ]
].copy()


Jumlah baris feat: 608


In [48]:
inset_feat_result.head(5)

,title,inset_neg,inset_pos,inset_compound,inset_polarity
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,0.800000,0.275000,-0.999734,-0.488372
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat",0.935802,0.214815,-0.999912,-0.626609
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,0.826446,0.512397,-0.998704,-0.234568
3,"8 WNI Dilaporkan Jadi Tentara Bayaran Rusia, Berapa Gaji Mereka?",0.733577,0.175182,-0.999680,-0.614458
4,"8 WNI Jadi Tentara Rusia, DPR Wanti-wanti Status Warga Negara Hilang",0.865471,0.260090,-0.999589,-0.537849


In [49]:
print(f"Jumlah baris lower: {len(news_base)}")
news_base[['title', 'inset_compound', 'inset_polarity']].head(5)

# Hasil sentimen InSet (lower_text)
inset_base_result = news_base[
    [
        "title",
        "inset_neg",
        "inset_pos",
        "inset_compound",
        "inset_polarity",
    ]
].copy()

inset_base_result.head(5)


Jumlah baris lower: 608


,title,inset_neg,inset_pos,inset_compound,inset_polarity
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,0.812133,0.244618,-0.999911,-0.537037
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat",0.895565,0.238913,-0.999964,-0.578815
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,0.758537,0.458537,-0.999505,-0.246493
3,"8 WNI Dilaporkan Jadi Tentara Bayaran Rusia, Berapa Gaji Mereka?",0.729545,0.168182,-0.999877,-0.625316
4,"8 WNI Jadi Tentara Rusia, DPR Wanti-wanti Status Warga Negara Hilang",0.841463,0.236585,-0.999878,-0.561086


In [50]:
inset_base_result.head(5)

,title,inset_neg,inset_pos,inset_compound,inset_polarity
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,0.812133,0.244618,-0.999911,-0.537037
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat",0.895565,0.238913,-0.999964,-0.578815
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,0.758537,0.458537,-0.999505,-0.246493
3,"8 WNI Dilaporkan Jadi Tentara Bayaran Rusia, Berapa Gaji Mereka?",0.729545,0.168182,-0.999877,-0.625316
4,"8 WNI Jadi Tentara Rusia, DPR Wanti-wanti Status Warga Negara Hilang",0.841463,0.236585,-0.999878,-0.561086


In [51]:
comparison = pd.DataFrame({
    "title": news["title"],
    "inset_lower": news_base["inset_compound"],
    "inset_lemma": news_feat["inset_compound"],
    "polarity_lower": news_base["inset_polarity"],
    "polarity_lemma": news_feat["inset_polarity"],
})

comparison["inset_diff"] = comparison["inset_lower"] - comparison["inset_lemma"]
comparison["polarity_diff"] = comparison["polarity_lower"] - comparison["polarity_lemma"]

comparison.head(5)


,title,inset_lower,inset_lemma,polarity_lower,polarity_lemma,inset_diff,polarity_diff
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,-0.999911,-0.999734,-0.537037,-0.488372,-0.000176,-0.048665
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat",-0.999964,-0.999912,-0.578815,-0.626609,-0.000052,0.047795
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,-0.999505,-0.998704,-0.246493,-0.234568,-0.000801,-0.011925
3,"8 WNI Dilaporkan Jadi Tentara Bayaran Rusia, Berapa Gaji Mereka?",-0.999877,-0.999680,-0.625316,-0.614458,-0.000197,-0.010859
4,"8 WNI Jadi Tentara Rusia, DPR Wanti-wanti Status Warga Negara Hilang",-0.999878,-0.999589,-0.561086,-0.537849,-0.000289,-0.023237


In [52]:
print("Perbedaan InSet compound :", (comparison["inset_diff"] != 0).sum())
print("Perbedaan InSet polarity :", (comparison["polarity_diff"] != 0).sum())


Perbedaan InSet compound : 608
Perbedaan InSet polarity : 608


### Keputusan final sumber teks untuk lexicon

- **InSet -> `lower_text`**: InSet adalah kamus kata Bahasa Indonesia yang berisi bentuk dasar dan sebagian bentuk berimbuhan. Teks `lower_text` masih memuat negasi ("tidak", "bukan", "belum") yang dipakai untuk membalik tanda kata sentimen, dan negasi ini hilang pada `text_lemmatized` karena termasuk stopword. Ini sejalan dengan alasan VADER -> `lower_text` pada versi CNBC.
- Cek hasil perbandingan di atas; bila selisih lemma vs lower besar, pertimbangkan ulang sebelum dikunci.
- **Loughran-McDonald tidak punya padanan Bahasa Indonesia**, jadi fitur `lm_*` tidak ada di versi ini.


In [53]:
# Kombinasi final: InSet dari lower_text
news_feat = compute_inset_sentiment(news, text_col='lower_text')

print(f"Jumlah baris: {len(news_feat)}")
news_feat[['title', 'inset_compound', 'inset_polarity']].head(5)


Jumlah baris: 608


,title,inset_compound,inset_polarity
0,Kenapa Pulau Larak Penting hingga Jadi Target Perang AS vs Iran?,-0.999911,-0.537037
1,"Eks Jenderal Israel Prihatin, Akui Pembersihan Etnis di Tepi Barat",-0.999964,-0.578815
2,Israel & Finlandia Diam-diam Perpanjang Kerja Sama Militer,-0.999505,-0.246493
3,"8 WNI Dilaporkan Jadi Tentara Bayaran Rusia, Berapa Gaji Mereka?",-0.999877,-0.625316
4,"8 WNI Jadi Tentara Rusia, DPR Wanti-wanti Status Warga Negara Hilang",-0.999878,-0.561086


# 1.2 Indeks Resiko Geopolitik berbasis Keyword

In [54]:
news_feat = compute_gpr_density(news_feat, text_col='lower_text', keywords=DEFAULT_GPR_KEYWORDS)

print(news_feat['gpr_hits'].describe())
news_feat[['title', 'gpr_hits', 'gpr_density']].sort_values('gpr_density', ascending=False).head(5)

count    608.000000
mean       1.636513
std        3.616425
min        0.000000
25%        0.000000
50%        0.000000
75%        2.000000
max       33.000000
Name: gpr_hits, dtype: float64


,title,gpr_hits,gpr_density
560,Drama Labil Tarif Trump: Dari 10% Naik Jadi 15% dalam Sehari,25,0.050505
492,"Hampir 50 Tahun Embargo, Diserang AS-Israel, Kenapa Iran Masih Kuat?",28,0.046434
564,Alasan Trump Luncurkan Tarif Dagang AS 10% Imbas 'Patah Hati',20,0.038241
285,Operator Terminal Minyak RI Kena Sanksi Uni Eropa Terkait Rusia,25,0.037707
562,"Baru Sehari 10%, Trump Kembali Kerek Tarif Dagang AS ke 15%",18,0.037037


# 1.3 TF-IDF + SVD

In [55]:
extractor = TfidfSvdExtractor(max_features=5000, n_components=75, min_df=5)
extractor.fit(news_feat['text_lemmatized'])

svd_vec = extractor.transform(news_feat['text_lemmatized'])
news_feat = pd.concat([news_feat.reset_index(drop=True), svd_vec.reset_index(drop=True)], axis=1)

print(f"Explained variance ratio (kumulatif, {extractor.n_components} komponen): {extractor.explained_variance_ratio_.sum():.3f}")
svd_vec.head(3)

Explained variance ratio (kumulatif, 75 komponen): 0.361


,tfidf_svd_0,tfidf_svd_1,tfidf_svd_2,tfidf_svd_3,tfidf_svd_4,tfidf_svd_5,tfidf_svd_6,tfidf_svd_7,tfidf_svd_8,tfidf_svd_9,...,tfidf_svd_65,tfidf_svd_66,tfidf_svd_67,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74
0,0.328834,0.164667,-0.162445,0.114688,0.161787,-0.004735,0.010176,0.138471,-0.021588,-0.021088,...,0.006284,-0.023557,0.024471,-0.014612,0.032494,-0.037311,-0.020953,-0.017803,0.006209,-0.036013
1,0.252315,-0.031594,0.271546,-0.073013,0.023189,-0.039426,0.087218,0.039867,-0.082000,0.016161,...,0.027518,-0.020480,0.081499,-0.011604,0.103253,0.017678,0.037199,0.094451,0.024934,-0.062020
2,0.213751,-0.025764,0.055014,-0.155620,0.021776,0.035556,-0.108016,0.002849,-0.027375,-0.026615,...,0.033996,0.082093,-0.002971,-0.065661,0.000353,-0.005077,-0.070211,-0.087780,0.003029,-0.030070


# 1.4 Agregaris ke Level Harian & Merge ke Data Kurs

daily_features = aggregate_daily_features(news_feat, date_col='aligned_trading_date')
print(f"Jumlah hari trading dengan fitur NLP: {len(daily_features)}")
daily_features.head(5)

In [56]:
daily_features = aggregate_daily_features(
    news_feat,
    date_col="aligned_trading_date",
)

print(f"Jumlah hari trading dengan fitur NLP: {len(daily_features)}")
daily_features.head()

Jumlah hari trading dengan fitur NLP: 141


,aligned_trading_date,inset_pos,inset_neg,inset_compound,inset_polarity,tfidf_svd_0,tfidf_svd_1,tfidf_svd_2,tfidf_svd_3,tfidf_svd_4,...,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74,gpr_density,gpr_hits,news_count
0,2026-01-23,0.260577,0.812599,-0.999847,-0.513108,0.213311,0.043391,0.068853,-0.140675,0.001433,...,0.024734,-0.023288,0.059270,-0.039794,-0.010033,0.061562,-0.028876,0.002513,2,2
1,2026-01-26,0.300543,0.752956,-0.999802,-0.425324,0.232834,0.022677,0.047743,-0.159289,0.025796,...,0.023439,0.013254,0.008472,0.007261,-0.048691,0.013432,-0.018235,0.000457,3,6
2,2026-01-27,0.272645,0.686695,-0.999718,-0.449496,0.227020,0.012234,0.029192,-0.153383,0.016431,...,0.028262,0.020208,0.029918,0.016378,0.088108,-0.020362,-0.066761,0.019492,23,3
3,2026-01-28,0.268769,0.837716,-0.999905,-0.514414,0.302482,0.061765,0.046159,0.030560,-0.006492,...,-0.028951,-0.014132,0.058075,-0.009397,-0.066913,0.007990,-0.032098,0.000000,0,3
4,2026-01-29,0.312612,0.740852,-0.999663,-0.411994,0.306992,0.107416,-0.113121,0.057811,0.024581,...,-0.017624,-0.026741,-0.023580,-0.038485,0.029768,0.012665,-0.017481,0.001188,1,2


In [57]:
merged_data_v2 = pd.merge(
    bi_raw,
    daily_features,
    left_on='Tanggal_dt',
    right_on='aligned_trading_date',
    how='left',
)

print(f"Baris merged_data_v2: {len(merged_data_v2)}")
print(f"Hari tanpa berita (NaN fitur NLP): {merged_data_v2['news_count'].isna().sum()}")
merged_data_v2.head(5)

Baris merged_data_v2: 1202
Hari tanpa berita (NaN fitur NLP): 1076


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,aligned_trading_date,inset_pos,...,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74,gpr_density,gpr_hits,news_count
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01,2026-09-01,0.269367,...,0.010850,0.043940,0.003029,-0.021517,0.018072,-0.021745,-0.000563,0.000000,0.0,5.0
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,2026-08-31,0.295354,...,-0.018949,0.005155,0.006387,-0.028353,0.010835,0.005295,-0.010698,0.008737,44.0,6.0
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,2026-08-28,0.272351,...,-0.005586,-0.031586,-0.053236,-0.014100,-0.004835,0.059860,-0.024022,0.014924,21.0,2.0
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM,2026-08-26,2026-08-26,0.258391,...,0.007160,-0.022514,0.005823,-0.021865,-0.005061,-0.002551,-0.022927,0.007324,28.0,7.0


# 1.5 Sanity Check

In [58]:
check = merged_data_v2.sort_values('Tanggal_dt').copy()
check['kurs_next'] = check['Kurs Jual'].shift(-1)
check['kurs_change_next'] = check['kurs_next'] - check['Kurs Jual']

corr_cols = ['inset_compound', 'inset_polarity', 'gpr_density', 'gpr_hits', 'news_count', 'kurs_change_next']
check[corr_cols].corr()['kurs_change_next']

inset_compound      0.147948
inset_polarity      0.184633
gpr_density        -0.063170
gpr_hits           -0.046312
news_count          0.026820
kurs_change_next    1.000000
Name: kurs_change_next, dtype: float64

In [59]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "dataset_processed_cnn_bi.csv"
merged_data_v2.to_csv(output_path, index=False)

print(f"File berhasil disimpan di: {output_path.resolve()}")

File berhasil disimpan di: C:\Users\ASUS\Downloads\code\nlpproject-grestercinta\data\processed\dataset_processed_cnn_bi.csv


In [61]:
#cek shape
merged_data_v2.shape

(1202, 91)